# Output 6 — Probability approach

Edit Macro/Ext, Input 6/7, CI Summary, and Probability covariates in Excel,
save, then reload.

Paths: baseline, A1 historical, and Chart Data most-extreme (MX) shock.
Distress probabilities use Excel `NORMDIST` covariates from `load_probability`.

See `docs/06-probability.qmd`.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from lic_dsf.load import load_core, load_probability, load_rating, load_stress
from lic_dsf.output import (
    external_debt_scenarios_table,
    probabilities_table,
    probability_panel,
)
from lic_dsf.rating import most_extreme_shock_id
from lic_dsf.scenario import ProbabilityAssumptions
from lic_dsf.stress import run_a1_historical_external, run_standard_external_stress

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent
WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"
WORKBOOK


In [ ]:
# Quick path: compute Output 6.
from lic_dsf.run import SHEET_6, compute_outputs

output_6 = compute_outputs(WORKBOOK, include=[SHEET_6])
output_6.sheets[SHEET_6]

In [ ]:
macro, external, ext_base, _pub_base = load_core(WORKBOOK)
rating = load_rating(WORKBOOK)
stress = load_stress(WORKBOOK)
ci = rating.ci

external_stress = run_standard_external_stress(
    macro, external, stress.input6, stress.residual
)
historical = run_a1_historical_external(macro, external, stress.residual)

first = int(macro.inputs.first_projection_year)
rating_years = list(range(first, first + 11))
panel_years = [int(y) for y in ext_base.years if int(y) >= first]
mx_sid = most_extreme_shock_id(
    {sid: book.pv_ppg_external_to_gdp() for sid, book in external_stress.items()},
    ci.thresholds.pv_debt_to_gdp,
    rating_years,
)
ci.country, mx_sid, panel_years[0], panel_years[-1]


## Output 6 panels

For each indicator: scenario levels (+ threshold bands) and NORMDIST probabilities.


In [ ]:
INDICATORS = (
    ("PV of debt-to-GDP ratio", "pv_ppg_external_to_gdp", "pv_debt_to_gdp"),
    ("PV of debt-to-exports ratio", "pv_ppg_external_to_exports", "pv_debt_to_exports"),
    ("Debt service-to-exports ratio", "ppg_debt_service_to_exports", "debt_service_to_exports"),
    ("Debt service-to-revenue ratio", "ppg_debt_service_to_revenue", "debt_service_to_revenue"),
)

assumptions = ProbabilityAssumptions(bandwidth=0.1)
covariates = load_probability(WORKBOOK)
thresh = ci.thresholds.as_dict()
mx_book = external_stress[mx_sid]

out_6: dict[str, pd.DataFrame] = {}
for title, method, indicator in INDICATORS:
    panel = probability_panel(
        {
            "baseline": getattr(ext_base, method)().reindex(panel_years),
            "historical": getattr(historical, method)().reindex(panel_years),
            "mx_shock": getattr(mx_book, method)().reindex(panel_years),
        },
        float(thresh[indicator]),
        indicator=indicator,
        assumptions=assumptions,
        covariates=covariates,
    )
    out_6[indicator] = panel
    display(Markdown(f"### {title}"))
    display(Markdown("**External debt scenarios**"))
    display(external_debt_scenarios_table(panel))
    display(Markdown("**Probabilities**"))
    display(probabilities_table(panel))
